# Demo - Q-Learning with FrozenLake


In this lab, let's use Q-learning to train an agent to play FrozenLake.




## 0. Install and Import Libraries
- `gymnasium`: provides the FrozenLake environment.
- `numpy`: stores and updates the Q-table.

In [2]:
# If needed, uncomment the following line:
# !pip install gymnasium

import gymnasium as gym
import numpy as np

[Gymnasium](https://gymnasium.farama.org/) provides environments for reinforcement learning. In this lab, we use its **FrozenLake-v1** environment.

If `import gymnasium` fails in Google Colab, run `!pip install gymnasium` in a new cell and then retry the import.

In [3]:
envs = gym.envs.registry
sorted(envs.keys())[:15] + ["..."]

['Acrobot-v1',
 'Ant-v2',
 'Ant-v3',
 'Ant-v4',
 'Ant-v5',
 'BipedalWalker-v3',
 'BipedalWalkerHardcore-v3',
 'Blackjack-v1',
 'CarRacing-v3',
 'CartPole-v0',
 'CartPole-v1',
 'CliffWalking-v1',
 'CliffWalkingSlippery-v1',
 'FrozenLake-v1',
 'FrozenLake8x8-v1',
 '...']

## 1. Load FrozenLake Environment

We start with a 4×4 FrozenLake environment and **non-slippery** (deterministic) movement.
Later, change `IS_SLIPPERY = True` and retrain to explore stochastic transitions.

**Note:** The Python action IDs are `0=Left, 1=Down, 2=Right, 3=Up`. These IDs differ from the `a1/a2/a3/a4` labels on the lecture diagrams.


In [4]:
IS_SLIPPERY = False

env = gym.make("FrozenLake-v1", is_slippery=IS_SLIPPERY)
num_states = env.observation_space.n
num_actions = env.action_space.n

action_names = {0: "Left", 1: "Down", 2: "Right", 3: "Up"}

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
for action_id, action_name in action_names.items():
    print(action_id, "=", action_name)


Observation space: Discrete(16)
Action space: Discrete(4)
0 = Left
1 = Down
2 = Right
3 = Up


## 2. Set Hyperparameters

- `alpha = 0.2` controls the size of each Q-value update.
- `gamma = 0.95` determines the weight of estimated future rewards.
- `epsilon = 0.3` means a **30% chance of sampling a random action** and a 70% chance of choosing a current greedy action. A random action can coincidentally be greedy.
- `seed = 42` is used to make the demonstration reproducible.


In [5]:
num_episodes = 5000
alpha = 0.2
gamma = 0.95
epsilon = 0.3
print("Maximum steps per episode:", env.spec.max_episode_steps)


Maximum steps per episode: 100


## 3. The Q-Learning Update Formula

For an **observed transition with a non-terminal next state**:

$$y = r + \gamma \max_{a'} Q(s', a')$$

For a **terminal next state**, $y=r$.

In both cases:

$$Q(s,a) \leftarrow Q(s,a)+\alpha\left[y-Q(s,a)\right]$$

The target uses an *observed reward* and an *estimated future value*; it is not necessarily the final true value.


## 4. Train the Agent

Initialize the Q-table to zeros. For each episode:

1. Reset the environment and choose each action using ε-greedy.
2. Observe the next state, reward, and termination/truncation flags.
3. Calculate the correct terminal/non-terminal target; update the Q-value for the **action actually taken**.
4. Repeat until termination or the 100-step limit; record training results.

**Important:** The training success rate includes exploratory actions. It is not the same as the performance of the learned greedy policy.


In [6]:
# Initialize the Q-table and training metrics.
Q_values = np.zeros((num_states, num_actions))

#random decisions for epsilon-greedy
rng = np.random.default_rng(seed=42)
env.action_space.seed(seed=42)

for iteration in range(num_episodes):
    state, info = env.reset(seed=42 if iteration == 0 else None)
    episode_return = 0

    for step in range(env.spec.max_episode_steps):
        # 1. Behavior: epsilon-greedy action selection
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q_values[state]))

        # 2. Observe an actual transition
        next_state, reward, terminated, truncated, info = env.step(action)

        # 3. Q-learning target: terminal states have no future return
        if terminated:
            target = reward
        else:
            target = reward + gamma * np.max(Q_values[next_state])

        # 4. Q-value update for the action that was actually taken
        Q_values[state, action] += alpha * (target - Q_values[state, action])
        state = next_state
        episode_return += reward

        if terminated or truncated:
            break

## 5. Inspect and Evaluate the Learned Q-Table
During evaluation, we use the learned Q-table to follow a **greedy policy**, without further exploration or Q-value updates.

After training, let's see what the agent has learned.
1. **Q-table:** Inspect the learned Q-values for each state-action pair.
2. **Greedy Policy:** Choose the action with the highest Q-value in each state. 
3. **Test the Policy:** Run one episode without exploration or Q-value updates (`HOLE` and `GOAL` denote terminal squares; those are not states in which the agent should select another action)
4. **Evaluation:** Examine the path, number of steps, and discounted return.

**Think about it:** Is one successful episode enough to conclude that the agent has learned a good policy? What other evaluation methods could we use?

In [ ]:
# 1. Inspect the learned Q-table
np.set_printoptions(precision=3, suppress=True)
print("Learned Q-table:")
print(Q_values)

# 2. Find the greedy action for each state
best_actions = np.argmax(Q_values, axis=1)

# Display the greedy policy as a 4x4 grid
map_tiles = env.unwrapped.desc.astype(str)

policy = np.array([
    action_names[a] for a in best_actions
], dtype=object).reshape(map_tiles.shape)

# Label terminal states
policy[map_tiles == "H"] = "HOLE"
policy[map_tiles == "G"] = "GOAL"

print("\nLearned Greedy Policy:")
print(policy)


Learned Q-table:
[[0.735 0.774 0.698 0.735]
 [0.735 0.    0.663 0.698]
 [0.698 0.677 0.238 0.282]
 [0.496 0.    0.    0.   ]
 [0.774 0.815 0.    0.735]
 [0.    0.    0.    0.   ]
 [0.    0.902 0.    0.607]
 [0.    0.    0.    0.   ]
 [0.815 0.    0.857 0.774]
 [0.815 0.902 0.902 0.   ]
 [0.857 0.95  0.    0.857]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.902 0.95  0.857]
 [0.902 0.95  1.    0.902]
 [0.    0.    0.    0.   ]]

Learned Greedy Policy:
[['Down' 'Left' 'Left' 'Left']
 ['Down' 'HOLE' 'Down' 'HOLE']
 ['Right' 'Down' 'Down' 'HOLE']
 ['HOLE' 'Right' 'Right' 'GOAL']]


In [11]:

# 3. Follow the learned greedy policy to generate a path
state, info = env.reset()
path = [state]

for step in range(env.spec.max_episode_steps):
    action = best_actions[state]
    next_state, reward, terminated, truncated, info = env.step(action)

    path.append(next_state)
    state = next_state

    if terminated or truncated:
        break

print("Path:", path)

# 4. Evaluate the path
steps = len(path) - 1
reached_goal = path[-1] == 15 # whether reached the goal

if reached_goal:
    discounted_return = gamma ** (steps - 1)
else:
    discounted_return = 0.0

print("Steps:", steps)
print("Reached Goal:", reached_goal)
print("Discounted Return:", round(discounted_return, 4))

Path: [0, 4, 8, 9, 13, 14, 15]
Steps: 6
Reached Goal: True
Discounted Return: 0.7738


## Discussion Questions
1. how do we know whether our agent has really learned?
2. what happens if we change epsilon?
3. what if we change alpha, our learning rate?
4. What if the lake becomes slippery?

